# Create Base Version of dataset 

Files kind:
- md_content : Markdown ocr with everything
- md_content_nohf : Markdown ocr without Header and Footer
- file_path : Address of PDF file
- layout_info : Bounding box of images and texts within a file
- layout_image : Bounding box of images and texts within a page


### Changes to v0.1

- In this version I removed tables and images from dataset.
- Text only version of dataset

In this notebook I exported 2 datasets : 
- all dataset with md_content_no_img_tbl
- dataset that only include md_content and md_content_no_img_tbl

In [1]:
from datasets import disable_caching
disable_caching()

/home/parsa/.conda/envs/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import json
import re
from datasets import load_from_disk


# Dataset path
ds_base_path = "./hf-dataset/panasonic-v0.0"

ds = load_from_disk(ds_base_path)

In [3]:
ds[0]['md_content']

'ERTJ1VR683J R-T Characteristics\n\n(for reference)\n\n$$R_{25} = 68 \\text{ kohm} \\quad \\text{+/-5\\%}$$\n\n$$B_{25/50} = 4250 \\text{ K} \\quad \\text{+/-2\\%}$$'

In [4]:
# Define a function to remove images and tables

import re

def remove_images(markdown_content):
    if not markdown_content:
        return ""

    # Remove standard markdown images: ![alt](url)
    content = re.sub(r'!\[[^\]]*\]\([^)]*\)', '', markdown_content)

    # Remove base64 images: ![](data:image/...)
    content = re.sub(
        r'!\[\]\(data:image/[^;]+;base64,[A-Za-z0-9+/=]+\)',
        '',
        content
    )

    # Clean extra blank lines
    content = re.sub(r'\n\s*\n', '\n\n', content).strip()

    return content

In [5]:
def process_example(example):
    md_content=example.get('md_content')
    md_content_no_img_tbl= remove_images(md_content)
    return {
        'md_content_no_img_tbl': md_content_no_img_tbl
    }

ds_processed = ds.map(process_example)

Map: 100%|██████████| 7525/7525 [00:02<00:00, 2568.29 examples/s]


In [5]:
ds_processed.save_to_disk("./hf-dataset/panasonic-no-img-tbl-v0.1/")

Saving the dataset (3/3 shards): 100%|██████████| 7525/7525 [00:00<00:00, 17713.88 examples/s]


In [6]:
columns_to_remove = [col for col in ds_processed.column_names if col not in ['md_content', 'md_content_no_img_tbl']]

ds_filtered = ds_processed.remove_columns(columns_to_remove)

ds_filtered[0]

{'md_content': 'ERTJ1VR683J R-T Characteristics\n\n(for reference)\n\n$$R_{25} = 68 \\text{ kohm} \\quad \\text{+/-5\\%}$$\n\n$$B_{25/50} = 4250 \\text{ K} \\quad \\text{+/-2\\%}$$',
 'md_content_no_img_tbl': 'ERTJ1VR683J R-T Characteristics\n\n(for reference)\n\n$$R_{25} = 68 \\text{ kohm} \\quad \\text{+/-5\\%}$$\n\n$$B_{25/50} = 4250 \\text{ K} \\quad \\text{+/-2\\%}$$'}

In [7]:
ds_filtered.save_to_disk("./hf-dataset/panasonic-only-md-no-img-v0.1/")

Saving the dataset (3/3 shards): 100%|██████████| 7525/7525 [00:00<00:00, 17068.21 examples/s]
